
# 📊 Analyse Exploratoire des Données (EDA) & Feature Engineering

```text
==========================================================================
TITRE       : Notebook d'Analyse Exploratoire (Churn Telco)
AUTEUR      : Adam Beloucif
PROJET      : Évaluation Finale Data Science Python
DATE        : Février 2026
DESCRIPTION : Comprendre pourquoi les clients résilient et préparer les données.
==========================================================================
```

> **🎯 Objectif Métier :** Détecter les signaux faibles de résiliation (Churn) et segmenter la base pour proposer des offres de rétention ciblées. 
> 
> *Règles appliquées : Pédagogie, Visualisation Pro, Gestion des types et des Outliers.*



## 1. 🛠️ Configuration & Bibliothèques
*POURQUOI ?* : Nous installons l'environnement. On force le style visuel de `seaborn` pour que les graphiques soient professionnels et prêts pour un rapport. On gère aussi les alertes (warnings) pour garder le notebook propre.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

# Configuration du style visuel pro
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
# Désactivation des warnings inutiles pour la propreté du notebook
warnings.filterwarnings('ignore')

print("✅ Bibliothèques chargées et environnement configuré.")


══════════════════════════════════════════════════════════════════════════
## 2. 📂 Chargement des Données & Correction des Types
──────────────────────────────────────────────────────────────────────────

*POURQUOI ?* : Le dataset brut a souvent des défauts. Typiquement, dans le dataset Telco, la colonne `TotalCharges` est vue comme un objet (texte) à cause de clients très récents n'ayant pas encore payé (chaîne vide `' '`). Nous devons impérativement la convertir en float pour pouvoir l'analyser mathématiquement, sous peine de faire planter nos modèles.


In [ ]:
# Chargement (Ajuster le chemin si nécessaire)
try:
    df = pd.read_csv('data/telco_churn.csv')
    print(f"📦 Dataset chargé : {df.shape[0]} lignes et {df.shape[1]} colonnes.")
except FileNotFoundError:
    print("❌ Erreur : Fichier introuvable. Assurez-vous d'être à la racine du projet.")

# Suppression de la clé primaire qui n'a aucune valeur prédictive
df.drop('customerID', axis=1, inplace=True)

# 🚨 Correction de la colonne 'TotalCharges' (Erreur classique de Kaggle)
# coerce force les chaines non convertibles en NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Pour les nouveaux clients (tenure = 0), ils n'ont pas encore payé
missing_charges = df['TotalCharges'].isnull().sum()
print(f"⚠️ Valeurs manquantes dans TotalCharges : {missing_charges} (remplacées par 0)")
df['TotalCharges'].fillna(0, inplace=True)

# Aperçu
display(df.head())


══════════════════════════════════════════════════════════════════════════
## 3. 🎯 Analyse de la Variable Cible (Churn)
──────────────────────────────────────────────────────────────────────────

*POURQUOI ?* : Avant de prédire, il faut observer ce qu'on veut prédire. On cherche ici à savoir si on a affaire à un problème de classes déséquilibrées (Imbalanced Data). Si la majorité des gens restent (Non-Churn), notre modèle pourrait "tricher" en prédisant tout le temps "Non".


In [ ]:
plt.figure(figsize=(6, 5))
ax = sns.countplot(x='Churn', data=df, palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
plt.title('Distribution de la Résiliation (Churn)', fontweight='bold')
plt.ylabel('Nombre de clients')

# Ajout des pourcentages sur les barres
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom', fontweight='bold')

plt.show()

print("🔍 CONCLUSION MÉTIER :")
print("Nous avons un déséquilibre modéré (26.5% de Churn). Lors de la modélisation, il faudra potentiellement")
print("utiliser l'argument `class_weight='balanced'` dans le Random Forest pour ne pas biaiser l'apprentissage.")


══════════════════════════════════════════════════════════════════════════
## 4. 📈 Profil du "Churner" - Facteurs de Résiliation
──────────────────────────────────────────────────────────────────────────

*POURQUOI ?* : C'est le cœur du métier. Le Machine Learning c'est bien, mais comprendre le business, c'est mieux. On croise ici le taux de désabonnement avec des variables clés comme le type de contrat et la présence du support technique.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Impact du Contrat
sns.histplot(data=df, x='Contract', hue='Churn', multiple='fill', shrink=.8, ax=axes[0], palette={'No': '#bdc3c7', 'Yes': '#c0392b'})
axes[0].set_title('Taux de Churn selon le Contrat', fontweight='bold')
axes[0].set_ylabel('Proportion')

# Impact du TechSupport
sns.histplot(data=df, x='TechSupport', hue='Churn', multiple='fill', shrink=.8, ax=axes[1], palette={'No': '#bdc3c7', 'Yes': '#c0392b'})
axes[1].set_title('Taux de Churn vs Support Technique', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("🚨 INSIGHTS MÉTIER :")
print("1. Danger immédiat sur le 'Month-to-month' : Ces contrats sans engagement fuient massivement.")
print("2. L'absence de support technique aggrave le churn. Vendre l'option 'TechSupport' fidélise fortement.")


══════════════════════════════════════════════════════════════════════════
## 5. ⚙️ Ingénierie des Caractéristiques (Feature Engineering)
──────────────────────────────────────────────────────────────────────────

*POURQUOI ?* : Les algorithmes adorent l'information synthétisée. Une donnée brute comme l'ancienneté en mois (`tenure`) peut être bruitée. En la regroupant par paliers ("groupes d'ancienneté"), on aide le modèle non-supervisé (Segmentation) et supervisé (Classification) à repérer des patterns plus évidents. 


In [ ]:
# Création des segments d'ancienneté
def group_tenure(month):
    if month <= 12: return '0-1 an'
    elif month <= 24: return '1-2 ans'
    elif month <= 48: return '2-4 ans'
    else: return '4+ ans'

df['Tenure_Group'] = df['tenure'].apply(group_tenure)

# Vérification rapide de la nouvelle variable
plt.figure(figsize=(8, 4))
sns.countplot(x='Tenure_Group', hue='Churn', data=df, order=['0-1 an', '1-2 ans', '2-4 ans', '4+ ans'], palette='dark:salmon')
plt.title('Churn par groupe d\'ancienneté', fontweight='bold')
plt.show()

print("🔨 Feature Engineering appliqué avec succès : Création de `Tenure_Group`.")
print("La prochaine étape sera l'encodage de toutes ces variables pour le Machine Learning.")